# 01 - Data Fetching

Fetch historical price data for stocks, ETFs, and crypto using Yahoo Finance.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.graph_objects as go
from src.data import fetch_prices, compute_returns, fetch_info

## Define Your Assets

Use Yahoo Finance tickers. Examples:
- US Stocks: `AAPL`, `MSFT`, `GOOGL`
- European ETFs: `VWCE.MI` (Vanguard FTSE All-World), `SXR8.DE` (iShares S&P 500)
- Crypto: `BTC-USD`, `ETH-USD`
- Indices: `^GSPC` (S&P 500), `^STOXX50E` (Euro Stoxx 50)

In [2]:
tickers = [
    "AAPL",
    "VWCE.MI",
    "BTC-USD",
    "^GSPC",
]

start_date = "2015-01-01"
end_date = "2025-01-01"

prices = fetch_prices(tickers, start=start_date, end=end_date)
print(f"Fetched {len(prices)} trading days for {len(tickers)} assets")
prices.tail()

Fetched 1232 trading days for 4 assets


Ticker,AAPL,BTC-USD,VWCE.MI,^GSPC
Date,,,,
2024-12-19,248.432877,97490.953125,133.820007,5867.080078
2024-12-20,253.107361,97755.929688,133.960007,5930.850098
2024-12-23,253.883102,94686.242188,133.759995,5974.069824
2024-12-27,254.201355,94164.859375,134.279999,5970.839844
2024-12-30,250.829773,92643.210938,133.630005,5906.939941


In [3]:
returns = compute_returns(prices)
returns.describe()

Ticker,AAPL,BTC-USD,VWCE.MI,^GSPC
count,1231.000000,1231.000000,1231.000000,1231.000000
mean,0.001180,0.002774,0.000485,0.000569
std,0.020186,0.041158,0.010358,0.013546
min,-0.128647,-0.371695,-0.077649,-0.119841
25%,-0.008515,-0.016043,-0.003965,-0.005372
50%,0.001254,0.001046,0.001066,0.000907
75%,0.012117,0.021547,0.005803,0.007382
max,0.119808,0.211097,0.076685,0.093828


## Price Chart (Normalized)

All assets normalized to 100 at the start for easy comparison.

In [4]:
normalized = prices / prices.iloc[0] * 100

fig = go.Figure()
for col in normalized.columns:
    fig.add_trace(go.Scatter(x=normalized.index, y=normalized[col], name=col, mode='lines'))
fig.update_layout(
    title="Normalized Prices (Base = 100)",
    yaxis_title="Price (normalized)",
    xaxis_title="Date",
    template="plotly_white",
    height=600,
)
fig.show()

## Daily Returns Distribution

In [5]:
fig = go.Figure()
for col in returns.columns:
    fig.add_trace(go.Histogram(x=returns[col], name=col, opacity=0.5, nbinsx=100))
fig.update_layout(
    title="Daily Returns Distribution",
    xaxis_title="Daily Return",
    yaxis_title="Frequency",
    template="plotly_white",
    barmode='overlay',
    height=500,
)
fig.show()

## Correlation Matrix

In [6]:
corr = returns.corr()

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale='RdBu',
    zmin=-1, zmax=1,
    text=corr.values.round(2),
    texttemplate='%{text}',
))
fig.update_layout(title="Returns Correlation Matrix", template="plotly_white", height=500)
fig.show()

## Save Data for Other Notebooks

In [7]:
prices.to_parquet('../data/prices.parquet')
returns.to_parquet('../data/returns.parquet')
print("Saved to ../data/prices.parquet and ../data/returns.parquet")

Saved to ../data/prices.parquet and ../data/returns.parquet
